In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Model Training — Databricks MLflow tracking
# MAGIC Reads the Delta table produced by `01_run_etl_pipeline`, converts it to
# MAGIC pandas (train.py's pandas-based loader can't authenticate to abfss:// on
# MAGIC its own — Spark can, so we bridge the two here), and calls `train()` from
# MAGIC `src/train.py`.
# MAGIC
# MAGIC This version tracks everything to Databricks' own built-in MLflow —
# MAGIC nothing extra to install or configure on the cluster. Good for
# MAGIC experimenting and iterating. If you want the model registered in
# MAGIC **Azure ML** instead (so `deploy.py` can find it), use
# MAGIC `03_run_training_azureml_mlflow.py` instead of this one.

# COMMAND ----------

storage_account = "enterprisemlopssa"  # <-- change if yours differs
processed_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/asset_features"

# COMMAND ----------

# MAGIC %md
# MAGIC ## Import the training logic straight from the repo

# COMMAND ----------

import os
import sys

notebook_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(notebook_dir, ".."))
src_dir = os.path.join(repo_root, "src")
sys.path.insert(0, src_dir)

from train import train  # noqa: E402

# COMMAND ----------

# MAGIC %md
# MAGIC ## Bridge: Delta (Spark) -> local Parquet (pandas)

# COMMAND ----------

features_sdf = spark.read.format("delta").load(processed_path)
features_pdf = features_sdf.toPandas()
print(f"Loaded {len(features_pdf):,} rows from {processed_path}")

local_features_dir = "/local_disk0/tmp/enterprise-mlops"  # local disk on the cluster node, not DBFS
os.makedirs(local_features_dir, exist_ok=True)
local_features_path = f"{local_features_dir}/asset_features.parquet"
features_pdf.to_parquet(local_features_path)
print(f"Wrote local copy for training -> {local_features_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Point MLflow at Databricks' built-in tracking (workspace-path experiment)

# COMMAND ----------

import mlflow
mlflow.set_experiment("/Shared/enterprise-asset-risk")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Train

# COMMAND ----------

metrics, run_id = train(
    features_path=local_features_path,
    model_type="random_forest",
    model_dir="/local_disk0/tmp/enterprise-mlops-models",  # local disk on the cluster node, not DBFS; unused when MLflow is available
)

print("Run ID:", run_id)
print("Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Where to look
# MAGIC Left sidebar → **Experiments** → `/Shared/enterprise-asset-risk`. The model
# MAGIC is registered in Databricks' own Model Registry here, not Azure ML — use
# MAGIC `03_run_training_azureml_mlflow.py` if you need it in Azure ML for `deploy.py`.